In [20]:
import os, sqlite3

db_path = os.path.abspath("../data/target.db")
print("DB PATH:", db_path)

# If this errors, close DB viewers and restart kernel
if os.path.exists(db_path):
    os.remove(db_path)
    print("Deleted old DB file.")
else:
    print("No DB file found, creating fresh.")

# Recreate empty DB file
conn = sqlite3.connect(db_path)
conn.close()
print("Fresh DB created.")


DB PATH: /Users/vedheshas/Downloads/enterprise-data-migration /data/target.db
Deleted old DB file.
Fresh DB created.


In [21]:
import sqlite3, os

db_path = os.path.abspath("../data/target.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE migration_runs (
    run_id TEXT PRIMARY KEY,
    start_time TEXT NOT NULL,
    end_time TEXT,
    source_file TEXT NOT NULL,
    total_raw INTEGER,
    total_clean INTEGER,
    total_rejected INTEGER,
    status TEXT,
    notes TEXT
);
""")

cursor.execute("""
CREATE TABLE legacy_customers_raw (
    run_id TEXT NOT NULL,
    customer_id INTEGER,
    full_name TEXT,
    email TEXT,
    country TEXT,
    signup_date TEXT,
    credit_score TEXT,
    account_status TEXT,
    kyc_status TEXT,
    last_updated TEXT,
    balance TEXT,
    currency TEXT
);
""")

cursor.execute("""
CREATE TABLE customers_clean (
    run_id TEXT NOT NULL,
    customer_id INTEGER NOT NULL,
    full_name TEXT NOT NULL,
    email TEXT,
    country TEXT,
    signup_date TEXT,
    credit_score INTEGER,
    account_status TEXT NOT NULL,
    kyc_status TEXT,
    last_updated TEXT,
    balance REAL CHECK (balance >= 0),
    currency TEXT CHECK (currency IN ('USD','INR')),
    PRIMARY KEY (run_id, customer_id),
    UNIQUE (run_id, email)
);
""")

cursor.execute("""
CREATE TABLE customers_rejected (
    reject_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    customer_id INTEGER,
    full_name TEXT,
    email TEXT,
    country TEXT,
    signup_date_raw TEXT,
    signup_date_parsed TEXT,
    credit_score_raw TEXT,
    account_status TEXT,
    kyc_status TEXT,
    last_updated_raw TEXT,
    last_updated_parsed TEXT,
    balance_raw TEXT,
    balance_num REAL,
    currency TEXT,
    reject_reason TEXT NOT NULL
);
""")

conn.commit()

# Confirm tables exist
import pandas as pd
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))

conn.close()
print("Schema created successfully.")


                   name
0        migration_runs
1  legacy_customers_raw
2       customers_clean
3    customers_rejected
4       sqlite_sequence
Schema created successfully.


In [22]:
import pandas as pd, sqlite3, os, uuid
from datetime import datetime

run_id = str(uuid.uuid4())
start_time = datetime.utcnow().isoformat(timespec="seconds") + "Z"

raw = pd.read_csv("../data/legacy_customers.csv")
clean = pd.read_csv("../data/customers_clean.csv")
rejected = pd.read_csv("../data/customers_rejected.csv")

db_path = os.path.abspath("../data/target.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
INSERT INTO migration_runs (run_id, start_time, source_file, status, notes)
VALUES (?, ?, ?, ?, ?)
""", (run_id, start_time, "data/legacy_customers.csv", "RUNNING", "Load started"))
conn.commit()

print("Run started:", run_id)
print("CSV sizes raw/clean/rejected:", len(raw), len(clean), len(rejected))


Run started: 5504560e-8eff-42f3-81f7-3c3693140962
CSV sizes raw/clean/rejected: 1050 96 954


/var/folders/r7/8kj70tv15nlbdrzwtjj7p5xw0000gn/T/ipykernel_86159/1565364678.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow().isoformat(timespec="seconds") + "Z"


In [23]:
db_path = os.path.abspath("../data/target.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()


In [24]:
# RAW
raw_db = raw.copy()
raw_db.insert(0, "run_id", run_id)
raw_db.to_sql("legacy_customers_raw", conn, if_exists="append", index=False)

# CLEAN
clean_db = clean.copy()
clean_db.insert(0, "run_id", run_id)
clean_db.to_sql("customers_clean", conn, if_exists="append", index=False)

# REJECTED (safe mapping)
rej = rejected.copy()
rej = rej.rename(columns={"signup_date": "signup_date_raw", "credit_score": "credit_score_raw"})
rej.insert(0, "run_id", run_id)
rej.to_sql("customers_rejected", conn, if_exists="append", index=False)

print("Loaded to DB raw/clean/rejected:", len(raw_db), len(clean_db), len(rej))



Loaded to DB raw/clean/rejected: 1050 96 954


In [25]:
end_time = datetime.utcnow().isoformat(timespec="seconds") + "Z"

total_raw = cursor.execute("SELECT COUNT(*) FROM legacy_customers_raw WHERE run_id=?", (run_id,)).fetchone()[0]
total_clean = cursor.execute("SELECT COUNT(*) FROM customers_clean WHERE run_id=?", (run_id,)).fetchone()[0]
total_rejected = cursor.execute("SELECT COUNT(*) FROM customers_rejected WHERE run_id=?", (run_id,)).fetchone()[0]

cursor.execute("""
UPDATE migration_runs
SET end_time=?, total_raw=?, total_clean=?, total_rejected=?, status=?, notes=?
WHERE run_id=?
""", (end_time, total_raw, total_clean, total_rejected, "SUCCESS", "Load completed", run_id))
conn.commit()

print("RUN COMPLETE:", run_id)
print("raw/clean/rejected:", total_raw, total_clean, total_rejected)
print("Reconciliation PASS:", total_raw == (total_clean + total_rejected))

conn.close()


RUN COMPLETE: 5504560e-8eff-42f3-81f7-3c3693140962
raw/clean/rejected: 1050 96 954
Reconciliation PASS: True


/var/folders/r7/8kj70tv15nlbdrzwtjj7p5xw0000gn/T/ipykernel_86159/3481919213.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow().isoformat(timespec="seconds") + "Z"
